In [9]:
import re
from itertools import combinations
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz

base_path = Path.cwd()
preferred_input_path = base_path / 'salida' / 'results.csv'
legacy_input_path = base_path / 'salida' / 'resultado_unificado_sin_duplicados.csv'
matrix_path = base_path / 'matriz_de_puntuacion.csv'

input_path = preferred_input_path if preferred_input_path.exists() else legacy_input_path
if not input_path.exists():
    raise FileNotFoundError(
        'Input result file not found. Expected one of: '
        f'{preferred_input_path} or {legacy_input_path}'
    )

df = pd.read_csv(input_path)
matriz = pd.read_csv(matrix_path, sep=';', encoding='latin1')


def normalizar_texto(texto):
    texto = '' if pd.isna(texto) else str(texto).lower()
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()


def contiene_patrones(texto, patrones):
    texto_normalizado = normalizar_texto(texto)
    return int(any(patron in texto_normalizado for patron in patrones))


patrones_estudio_caso = [
    'case study',
    'case studies',
    'case based',
    'use case',
    'use cases',
    'industrial case',
    'industrial cases',
    'real world case',
    'real world cases',
    'empirical study',
    'field study',
    'pilot study',
    'proof of concept',
    'prototype validation',
    'estudio de caso',
    'estudios de caso',
    'caso de estudio',
    'casos de estudio',
    'estudo de caso',
    'estudos de caso',
    'prova de conceito',
]

print(f'Input records from result file: {len(df)}')
print(f'Using input file: {input_path}')

Input records from result file: 1441
Using input file: /home/d4ndres/Escritorio/revision sistematica/salida/results.csv


In [10]:
coeficientes = (
    matriz.iloc[:-1].copy()
    .rename(columns={matriz.columns[0]: 'metrica', matriz.columns[1]: 'coeficiente'})
)
coeficientes['coeficiente'] = pd.to_numeric(coeficientes['coeficiente'], errors='coerce').fillna(0)
coeficientes = dict(zip(coeficientes['metrica'], coeficientes['coeficiente']))

texto_busqueda = df['title'].fillna('') + ' ' + df['description'].fillna('')
anio_actual = pd.Timestamp.today().year
antiguedad = anio_actual - pd.to_numeric(df['pub_year'], errors='coerce')

df['metric_keyword_1'] = df['source'].eq('s1').astype(int)
df['metric_keyword_2'] = df['source'].eq('s2').astype(int)

df['metric_artigo_em_periodico'] = df['aggregationType'].fillna('').eq('Journal').astype(int)
df['metric_artigo_em_congresso'] = df['aggregationType'].fillna('').eq('Conference Proceeding').astype(int)
df['metric_livro'] = df['aggregationType'].fillna('').eq('Book').astype(int)

df['metric_artigo_de_pesquisa'] = df['subtypeDescription'].fillna('').eq('Article').astype(int)
df['metric_artigo_de_revisao_da_literatura'] = df['subtypeDescription'].fillna('').eq('Review').astype(int)
df['metric_estudo_de_caso'] = texto_busqueda.map(lambda texto: contiene_patrones(texto, patrones_estudio_caso))

df['metric_idade_ate_2_anos'] = antiguedad.le(2).fillna(False).astype(int)
df['metric_idade_de_2_a_5_anos'] = antiguedad.gt(2).fillna(False).mul(antiguedad.le(5).fillna(False)).astype(int)
df['metric_idade_mais_que_5_anos'] = antiguedad.gt(5).fillna(False).astype(int)

df['metric_citacoes_raw'] = pd.to_numeric(df['citedby_count'], errors='coerce').fillna(0)
df['metric_percentil_raw'] = pd.to_numeric(df['citescore_percentile'], errors='coerce').fillna(0)

max_citacoes = df['metric_citacoes_raw'].max()
max_percentil = df['metric_percentil_raw'].max()

df['metric_citacoes_normalized'] = df['metric_citacoes_raw'] / max_citacoes if max_citacoes else 0
df['metric_percentil_normalized'] = df['metric_percentil_raw'] / max_percentil if max_percentil else 0

In [11]:
mapeo_metricas = {
    'Keyword 1': 'metric_keyword_1',
    'Keyword 2': 'metric_keyword_2',
    'Estudo de caso': 'metric_estudo_de_caso',
    'Artigo de pesquisa': 'metric_artigo_de_pesquisa',
    'Percentil': 'metric_percentil_normalized',
    'Artigo em Periódico': 'metric_artigo_em_periodico',
    'Idade (até 2 anos)': 'metric_idade_ate_2_anos',
    'Artigo em Congresso': 'metric_artigo_em_congresso',
    'Idade (de 2 a 5 anos)': 'metric_idade_de_2_a_5_anos',
    'Citações': 'metric_citacoes_normalized',
    'Artigo de revisão da literatura': 'metric_artigo_de_revisao_da_literatura',
    'Idade (mais que 5 anos)': 'metric_idade_mais_que_5_anos',
    'Livro': 'metric_livro',
}

for metrica, columna in mapeo_metricas.items():
    nombre_columna_puntaje = f'score_{columna.removeprefix("metric_")}'
    df[nombre_columna_puntaje] = df[columna] * coeficientes.get(metrica, 0)

columnas_score = [col for col in df.columns if col.startswith('score_')]
df['score_final'] = df[columnas_score].sum(axis=1)
df = df.sort_values(by='score_final', ascending=False).reset_index(drop=True)

In [12]:
titulos = df[['title', 'doi', 'publicationName', 'pub_year', 'source']].copy()
titulos = titulos.dropna(subset=['title']).reset_index(names='row_id')
titulos['title_normalized'] = titulos['title'].map(normalizar_texto)

posibles_duplicados = []
umbral_similitud = 0.93

for izquierda, derecha in combinations(titulos.itertuples(index=False), 2):
    similitud = fuzz.ratio(izquierda.title_normalized, derecha.title_normalized) / 100
    if similitud >= umbral_similitud:
        posibles_duplicados.append({
            'similarity': round(similitud, 4),
            'row_id_1': izquierda.row_id,
            'row_id_2': derecha.row_id,
            'doi_1': izquierda.doi,
            'doi_2': derecha.doi,
            'title_1': izquierda.title,
            'title_2': derecha.title,
            'publication_1': izquierda.publicationName,
            'publication_2': derecha.publicationName,
            'year_1': izquierda.pub_year,
            'year_2': derecha.pub_year,
            'source_1': izquierda.source,
            'source_2': derecha.source,
        })

posibles_duplicados = pd.DataFrame(posibles_duplicados).sort_values(
    by='similarity', ascending=False
) if posibles_duplicados else pd.DataFrame()

In [13]:
print(f'Records loaded from result file: {len(df)}')
print(f'Possible duplicates by similar title: {len(posibles_duplicados)}')

columnas_metricas = [
    'source',
    'aggregationType',
    'subtypeDescription',
    'pub_year',
    'citedby_count',
    'citescore_percentile',
    'metric_keyword_1',
    'metric_keyword_2',
    'metric_artigo_em_periodico',
    'metric_artigo_em_congresso',
    'metric_livro',
    'metric_artigo_de_pesquisa',
    'metric_artigo_de_revisao_da_literatura',
    'metric_estudo_de_caso',
    'metric_idade_ate_2_anos',
    'metric_idade_de_2_a_5_anos',
    'metric_idade_mais_que_5_anos',
    'metric_citacoes_normalized',
    'metric_percentil_normalized',
    'score_final',
]

df['maintain'] = ''

display(df[['title', 'maintain'] + columnas_metricas + columnas_score].head(20))
display(posibles_duplicados.head(20))

output_score = base_path / 'salida' / 'resultado_puntuado.csv'
output_duplicates = base_path / 'salida' / 'posibles_duplicados_titulo.csv'

df.to_csv(output_score, index=False)
posibles_duplicados.to_csv(output_duplicates, index=False)

print(f'Scored output saved to: {output_score}')
print(f'Possible duplicates saved to: {output_duplicates}')

Records loaded from result file: 1441
Possible duplicates by similar title: 4


,title,maintain,source,aggregationType,subtypeDescription,pub_year,citedby_count,citescore_percentile,metric_keyword_1,metric_keyword_2,...,score_artigo_de_pesquisa,score_percentil_normalized,score_artigo_em_periodico,score_idade_ate_2_anos,score_artigo_em_congresso,score_idade_de_2_a_5_anos,score_citacoes_normalized,score_artigo_de_revisao_da_literatura,score_idade_mais_que_5_anos,score_livro
0,BIM-based preassembly analysis for design for ...,,s1,Journal,Article,2024,54,99.0,1,0,...,25,24.000000,13,12,0,0,0.642857,0,0,0
1,Assessing user performance in augmented realit...,,s1,Journal,Article,2024,46,99.0,1,0,...,25,24.000000,13,12,0,0,0.547619,0,0,0
2,Optimal layout planning for human robot collab...,,s1,Journal,Article,2024,70,97.0,1,0,...,25,23.515152,13,12,0,0,0.833333,0,0,0
3,The minimal AR authoring approach: Validation ...,,s1,Journal,Article,2024,14,99.0,1,0,...,25,24.000000,13,12,0,0,0.166667,0,0,0
4,Systematic AR-based assembly guidance for smal...,,s1,Journal,Article,2025,12,99.0,1,0,...,25,24.000000,13,12,0,0,0.142857,0,0,0
5,Streamlining Assembly Instruction Design (S-AI...,,s1,Journal,Article,2025,7,99.0,1,0,...,25,24.000000,13,12,0,0,0.083333,0,0,0
6,Proposal of a complexity model for human-robot...,,s1,Journal,Article,2025,6,99.0,1,0,...,25,24.000000,13,12,0,0,0.071429,0,0,0
7,Improving Work Skills in People with Disabilit...,,s1,Journal,Article,2024,3,99.0,1,0,...,25,24.000000,13,12,0,0,0.035714,0,0,0
8,Designing worker assistance systems–Methodolog...,,s1,Journal,Article,2025,3,99.0,1,0,...,25,24.000000,13,12,0,0,0.035714,0,0,0
9,A digital twin-enabled system for real-time in...,,s1,Journal,Article,2026,0,99.0,1,0,...,25,24.000000,13,12,0,0,0.000000,0,0,0


,similarity,row_id_1,row_id_2,doi_1,doi_2,title_1,title_2,publication_1,publication_2,year_1,year_2,source_1,source_2
1,1.000,789,1376,NaN,NaN,ACM International Conference Proceeding Series,ACM International Conference Proceeding Series,ACM International Conference Proceeding Series,ACM International Conference Proceeding Series,2017,2018,s1,s1
2,1.000,1108,1109,NaN,NaN,"14th International Conference on Virtual, Augm...","14th International Conference on Virtual, Augm...",Lecture Notes in Computer Science Including Su...,Lecture Notes in Computer Science Including Su...,2022,2022,s1,s1
3,1.000,1236,1237,NaN,NaN,23rd International Conference on Human-Compute...,23rd International Conference on Human-Compute...,Communications in Computer and Information Sci...,Communications in Computer and Information Sci...,2021,2021,s1,s1
0,0.939,355,993,10.1016/j.mfglet.2025.10.003,10.18260/1-2--55435,Analysis of user experience in extended realit...,Analysis of User Experience in Digital Reality...,Manufacturing Letters,ASEE Annual Conference and Exposition Conferen...,2025,2025,s1,s1


Scored output saved to: /home/d4ndres/Escritorio/revision sistematica/salida/resultado_puntuado.csv
Possible duplicates saved to: /home/d4ndres/Escritorio/revision sistematica/salida/posibles_duplicados_titulo.csv
